In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-2t69LdIulCeS


In [4]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

# FewShotPromptTemplate 설계

In [10]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 와인과 치즈
            통화 : 유로
            """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
            """
    },
    {
        "country": "한국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 서울
            언어 : 한국어
            음식 : 김치와 비빔밥
            통화 : 원
            """
    }
]

# 2단계 : 예시용 프롬프트 템플릿 정의

In [11]:
example_template = """
    Human : {country}
    AI : {answer}
    """

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human : {country}\n    AI : {answer}\n    ')

# 3단계 : FewShotPromptTemplate 생성 후 결합

In [14]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문 
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [15]:
chat = ChatOpenAI(temperature=0)

In [16]:
chain = prompt | chat

In [19]:
result = chain.invoke(
    {
        "country": "한국"
    }
)

In [23]:
result.content

'AI: \n            저는 이렇게 알고 있어요.\n            수도 : 서울\n            언어 : 한국어\n            음식 : 김치와 비빔밥\n            통화 : 원'

# FewShotChatPromptMessage

In [17]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [ ]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 와인과 치즈
            통화 : 유로
            """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
            """
    },
    {
        "country": "한국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 서울
            언어 : 한국어
            음식 : 김치와 비빔밥
            통화 : 원
            """
    }
]

# 2단계 예시용 프롬프트 템플릿 정의

In [36]:
example = ChatPromptTemplate.from_messages([
    ("human","{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}"),
])

In [37]:
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example,
    examples=examples
)

In [29]:
# final = prompt | chain

In [38]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요"),
    example_prompt,
    ("human","{country}에 대해서 어떻게 알고 있어요?")
])

In [39]:
final_chain = final_prompt | chat

In [40]:
result = final_chain.invoke({
    "country" : "일본"
})


In [43]:
print(result.content)

일본은 아시아 대륙 동쪽에 위치한 섬나라로, 수도는 도쿄이며 일본어가 공용어로 사용됩니다. 일본은 고대부터 현대까지 독특한 문화와 전통을 간직하고 있으며, 일본의 음식 문화는 세계적으로 유명합니다. 또한 일본은 선진 기술과 혁신적인 산업으로 유명하며, 일본의 역사와 예술은 세계적으로 인정받고 있습니다.


# LengthBasedExampleSelector 설계

In [44]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [45]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 와인과 치즈
            통화 : 유로
            """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
            """
    },
    {
        "country": "한국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 서울
            언어 : 한국어
            음식 : 김치와 비빔밥
            통화 : 원
            """
    }
]

In [46]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI : {answer}")

# Selector 연결

In [47]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제의 양 허용(토큰 개수)
    max_length= 10
)



In [49]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix = "Human: {country}에 대해서 어떻게 알고 있나요?",
    input_variables = ["country"]
)

In [51]:
print(prompt.format(**{
    "country": "브라질"
}))

Human: 브라질에 대해서 어떻게 알고 있나요?


In [52]:
result = final_chain.invoke({
    "country": "독일"
})

In [54]:
print(result.content)


            저는 이렇게 알고 있어요.
            수도 : 베를린
            언어 : 독일어
            음식 : 소세지와 맥주
            통화 : 유로


# Chat 모델(LengthBasedExampleSelector)

In [55]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 파리
            언어 : 프랑스어
            음식 : 와인과 치즈
            통화 : 유로
            """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 도쿄
            언어 : 일본어
            음식 : 초밥과 라멘
            통화 : 엔
            """
    },
    {
        "country": "한국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도 : 서울
            언어 : 한국어
            음식 : 김치와 비빔밥
            통화 : 원
            """
    }
]

In [59]:
length_prompt = PromptTemplate(
    input_variables = ["country", "answer"],
    template="""
        Human: {country}에 대해서 어떻게 알고 있어요? \n
        AI : {answer}
        """
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai","{answer}")
])

example_selector = LengthBasedExampleSelector(
    examples= examples,
    example_prompt= length_prompt,
    max_length=200
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector = example_selector,
    example_prompt=example_prompt
)

In [60]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system","당신은 지리학 전문가입니다"),
    fewshot_prompt,
    ("human","{country}에 대해 어떻게 알고 있어요?"),
])

In [61]:
chain = final_prompt | chat

In [62]:
result = chain.invoke({
    "country":"대한민국"
})

In [63]:
print(result.content)

대한민국에 대해 알고 있는 정보는 다음과 같습니다:
수도: 서울
언어: 한국어
인구: 약 5천만 명
경제: 세계적으로 경제력이 강한 나라로 발전
문화: 한류, 한식, 한국 전통 문화 등이 유명
지리: 한반도에 위치하여 북한과 접해 있음
정치: 공화제로 운영되며 대통령이 국가의 수장
경제: 선진국으로 발전하여 IT, 자동차, 조선업 등이 주요 산업
환경: 자연경관이 아름다우며 환경보호에 신경을 씀
